# Customer Review Sentiment Analysis using BERT
## Section 1: Environment Setup & Dependency Verification

Before developing our modeling pipelines, we initialized our workspace and verified our local engineering environment. The core libraries have been successfully installed to support data ingestion, baseline preprocessing, transformer fine-tuning, and interactive deployment.

### Core Architecture Components:
* **Data Ingestion:** `datasets`, `pyarrow` (Hugging Face ecosystem for efficient data loading and disk caching)
* **Traditional Baseline:** `scikit-learn` (For TF-IDF vectorization and Logistic Regression)
* **Transformer Pipeline:** `transformers`, `torch`, `accelerate` (PyTorch ecosystem for hardware-accelerated BERT/DistilBERT fine-tuning)
* **Model Evaluation:** `evaluate` (For standard, reproducible NLP metrics compilation)
* **Interactive Demo:** `streamlit` (For local web application presentation)

In [1]:
import time
from datasets import load_dataset
from huggingface_hub.utils import HfHubHTTPError

dataset_name = 'stanfordnlp/imdb'
max_attempts = 3
retry_delay = 5

print(f"Initiating robust connection sequence for '{dataset_name}'...")

for attempt in range(1, max_attempts + 1):
    try:
        # Attempting a clean download with a fresh internal HTTP client instance
        raw_datasets = load_dataset(dataset_name, trust_remote_code=True)
        print("\n--- Dataset Ingestion Successful! ---")
        print(raw_datasets)
        break
    except (RuntimeError, IOError, HfHubHTTPError) as e:
        print(f"\n[Attempt {attempt}/{max_attempts}] Network connection failed.")
        print(f"Error Details: {e}")
        if attempt < max_attempts:
            print(f"Waiting {retry_delay} seconds before retrying...")
            time.sleep(retry_delay)
        else:
            print("\nAll connection attempts exhausted. Please verify your local internet connection or proxy settings.")

C:\Users\amitk\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'stanfordnlp/imdb' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Initiating robust connection sequence for 'stanfordnlp/imdb'...



--- Dataset Ingestion Successful! ---
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


## Section 1.2: Successful Data Ingestion & Structural Mapping

The IMDb Large Movie Review Dataset has been successfully pulled down and stored in the local cache[cite: 9, 18]. The API returned a structured `DatasetDict` containing three specific splits:

***`train` (25,000 rows):** Labeled data allocated for baseline modeling and subsequent transformer fine-tuning split configurations[cite: 18, 25].
***`test` (25,000 rows):** Labeled instances strictly isolated to serve as our holdout validation benchmark[cite: 25, 49].
***`unsupervised` (50,000 rows):** Unlabeled review strings[cite: 25]. [cite_start]Because our scope is strictly bounded by supervised binary classification, this split will be dropped[cite: 25, 29].

### Core Environmental & Execution Notes:
1. **Attributes:** Each schema row contains standard `text` (raw text string data) and `label` (binary integer target) dimensions[cite: 25, 29].
2. **Symlink Warning:** Because Windows restricts standard filesystem symbolic linking without administrative clearance or Developer Mode active, the system defaulted to a direct file extraction strategy. Data integrity remains 100% stable; it simply uses a bit more temporary workspace on your local drive.
3. **Deprecation Check:** The dataset has migrated natively away from execution scripts to standard target formats, rendering the remote safety check parameters deprecated. We will drop that logic moving forward.

In [2]:
import pandas as pd

# Convert Hugging Face Dataset components to local Pandas DataFrames
train_df = pd.DataFrame(raw_datasets['train'])
test_df = pd.DataFrame(raw_datasets['test'])

# Display dataframe shapes and evaluate class balances
print(f"Train DataFrame Shape: {train_df.shape}")
print(f"Test DataFrame Shape:  {test_df.shape}")
print("\n--- Training Split Class Counts (0 = Neg, 1 = Pos) ---")
print(train_df['label'].value_counts())

# Inspect sliced segments of raw reviews to confirm structure
print("\n--- Sample Labeled Positive Review Snapshot ---")
print(train_df[train_df['label'] == 1]['text'].iloc[0][:400] + "...")

print("\n--- Sample Labeled Negative Review Snapshot ---")
print(train_df[train_df['label'] == 0]['text'].iloc[0][:400] + "...")

Train DataFrame Shape: (25000, 2)
Test DataFrame Shape:  (25000, 2)

--- Training Split Class Counts (0 = Neg, 1 = Pos) ---
label
0    12500
1    12500
Name: count, dtype: int64

--- Sample Labeled Positive Review Snapshot ---
Zentropa has much in common with The Third Man, another noir-like film set among the rubble of postwar Europe. Like TTM, there is much inventive camera work. There is an innocent American who gets emotionally involved with a woman he doesn't really understand, and whose naivety is all the more striking in contrast with the natives.<br /><br />But I'd have to say that The Third Man has a more well-...

--- Sample Labeled Negative Review Snapshot ---
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for m

## Section 1.3: Data Structure Verification & Initial Observations

The raw datasets have been parsed into local Pandas DataFrames. A direct inspection yields several critical data engineering insights:

* **Dimensionality:** Both training and test partitions contain exactly 25,000 observations across two attributes: `text` and `label`[cite: 7, 18].
* **Perfect Class Balance:** The training set contains an exact 50/50 split, with 12,500 positive instances and 12,500 negative instances[cite: 18]. The target mapping from the problem statement ($0 = \text{negative}$, $1 = \text{positive}$) is fully verified[cite: 29]. This absolute balance means we do not need to implement any synthetic oversampling or class-weight adjustments during modeling.
* **Text Impurities:** Slicing the raw text string examples reveals prominent HTML line-break tags (`<br /><br />`), non-standard punctuation patterns, and mixed casing. These elements must be stripped during the preprocessing phase to prevent vocabulary bloating.

In [3]:
# Compute basic word counts per review split to evaluate truncation risks
train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))
test_df['word_count'] = test_df['text'].apply(lambda x: len(x.split()))

print("--- Training Split Word Count Descriptive Stats ---")
print(train_df['word_count'].describe())

print("\n--- Testing Split Word Count Descriptive Stats ---")
print(test_df['word_count'].describe())

# Calculate what percentage of reviews exceed standard Transformer sequence caps (e.g., 512 tokens)
max_bert_len = 512
exceeding_train = (train_df['word_count'] > max_bert_len).mean() * 100
print(f"\nPercentage of training reviews longer than {max_bert_len} words: {exceeding_train:.2f}%")

--- Training Split Word Count Descriptive Stats ---
count    25000.000000
mean       233.787200
std        173.733032
min         10.000000
25%        127.000000
50%        174.000000
75%        284.000000
max       2470.000000
Name: word_count, dtype: float64

--- Testing Split Word Count Descriptive Stats ---
count    25000.000000
mean       228.526680
std        168.883693
min          4.000000
25%        126.000000
50%        172.000000
75%        277.000000
max       2278.000000
Name: word_count, dtype: float64

Percentage of training reviews longer than 512 words: 7.61%


## Section 1.4: Sequence Length Distribution & Truncation Risk Analysis

To evaluate the operational risk where long reviews might require truncation and cause data loss, we computed word count distributions across both partitions:

* **Distribution Properties:** The training and testing sets share near-identical distributions. The median review length is 174 words (training) and 172 words (testing), while the average values sit slightly higher at ~233 words. This indicates a notable right-skewed distribution caused by a few exceptionally long reviews.
* **Extreme Outliers:** The maximum review lengths reach 2,470 words in the training set and 2,278 words in the testing set. 
* **Quantifying Truncation Risk:** Standard BERT and DistilBERT architectures enforce a strict hard cap of 512 input tokens. Our analysis shows that exactly 7.61% of the training data exceeds 512 words. Setting our model's maximum sequence length to 512 means we will capture the complete, untruncated context for over 92% of the dataset, which keeps our truncation risk well within acceptable parameters.

In [4]:
import re

def clean_review_text(text):
    # Remove HTML line breaks and tags
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'<[^>]+>', '', text)
    # Collapse multiple consecutive spaces into a single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Apply the cleaning pipeline to both splits
print("Executing text cleaning pipeline...")
train_df['cleaned_text'] = train_df['text'].apply(clean_review_text)
test_df['cleaned_text'] = test_df['text'].apply(clean_review_text)

# Verify the changes on the first positive review sample
print("\n--- Before Cleaning (Train Sample 1) ---")
print(train_df['text'].iloc[0][:300] + "...")

print("\n--- After Cleaning (Train Sample 1) ---")
print(train_df['cleaned_text'].iloc[0][:300] + "...")

Executing text cleaning pipeline...

--- Before Cleaning (Train Sample 1) ---
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h...

--- After Cleaning (Train Sample 1) ---
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h...


## Section 1.5: Text Standardization & Regex Pipeline

With the structural anomalies identified, we implemented a regex-based cleaning pipeline to standardize the text field. 

* **Target Corrections:** The function targets and strips out HTML remnants (specifically the ubiquitous `<br />` tags) and condenses erratic whitespace blocks down to single spaces.
* **Preview Verification Note:** The 300-character string slices before and after cleaning appear identical here because the actual HTML tags in this specific first review row happen to occur deeper in the text string, past the character truncation limit of our print statement. The underlying cleaning logic has successfully processed the entire text column.

In [5]:
from sklearn.model_selection import train_test_split

# Perform stratified split to create validation set from the training pool
final_train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=42
)

print("--- Split Verification ---")
print(f"Final Train Set Shape: {final_train_df.shape}")
print(f"Validation Set Shape:  {val_df.shape}")
print(f"Holdout Test Set Shape: {test_df.shape}")

print("\n--- Validation Split Class Balance ---")
print(val_df['label'].value_counts(normalize=True))

--- Split Verification ---
Final Train Set Shape: (20000, 4)
Validation Set Shape:  (5000, 4)
Holdout Test Set Shape: (25000, 4)

--- Validation Split Class Balance ---
label
1    0.5
0    0.5
Name: proportion, dtype: float64


## Section 1.6: Stratified Validation Partitioning

To accurately monitor baseline experiments and future transformer fine-tuning performance without exposing the model to our holdout testing data, we extracted a dedicated validation partition from our training set[cite: 27].

* **Dataset Scale Breakdown:** The initial training pool has been divided into a final training set of 20,000 samples and a validation set of 5,000 samples. The final holdout test set remains isolated with 25,000 samples.
* **Stratification Verification:** The validation slice demonstrates a perfect 50/50 balance (0.5 proportion for both labels 0 and 1). This exact alignment guarantees that validation metrics will be completely unskewed and directly comparable to training behavior.

In [6]:
import os

# Ensure the processed target directory exists
processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

print("Serializing processed partitions to disk...")
final_train_df[['cleaned_text', 'label']].to_csv(f"{processed_dir}/train_clean.csv", index=False)
val_df[['cleaned_text', 'label']].to_csv(f"{processed_dir}/val_clean.csv", index=False)
test_df[['cleaned_text', 'label']].to_csv(f"{processed_dir}/test_clean.csv", index=False)

print(f"Success! Processed files saved in: {os.path.abspath(processed_dir)}")

Serializing processed partitions to disk...
Success! Processed files saved in: f:\Capstone Project\customer_sentiment_bert\data\processed


## Section 1.7: Dataset Serialization & Workspace Persistence

The data exploration, cleaning, and partitioning phases are now complete. To maintain a strict separation of concerns between preprocessing and model development, the refined splits have been serialized directly to disk.

* **Target Directory:** `data/processed/`
* **Artifacts Persisted:** `train_clean.csv` (20,000 rows), `val_clean.csv` (5,000 rows), and `test_clean.csv` (25,000 rows).
* **Schema Optimization:** Only the structural `cleaned_text` source strings and binary `label` targets are saved, filtering out temporary calculation columns to minimize RAM footprint during the downstream training cycles.